# Agentic Radiology Assistant

**Portfolio adaptation of an MSc Intelligent Systems coursework project.**

This notebook demonstrates a multi-agent chest X-ray workflow using Gemini, Gradio and a public Hugging Face dataset.

> **ACADEMIC DEMO ONLY - NOT FOR CLINICAL USE.** The outputs are generated by a general-purpose AI model and are not medically validated. Do not use this notebook for diagnosis, treatment, triage or other real-world clinical decisions.

### Portfolio maintenance changes

This version preserves the original project structure while migrating from the deprecated `google-generativeai` package to `google-genai`, removing saved temporary share links/personal identifiers, and relabelling the original batch "accuracy" test as a **workflow-consistency demonstration**. The original `agent_history(truth)` design is intentionally retained because it simulates the clinical history/referral context a radiologist would normally receive; however, because that context is seeded from the known dataset label, the original batch run is not a blinded diagnostic-accuracy test. This portfolio version also adds a separate blinded benchmark in which the dataset label is withheld from all inference-stage agents and used only after prediction for scoring.

In [ ]:
# Install dependencies in Colab / a fresh notebook environment.
!pip install -q google-genai gradio datasets

## 1. Imports and configuration

The API key is read from Colab Secrets/environment variables when available and otherwise requested interactively. It is never stored in the notebook.

In [ ]:
import os
import time
import random
import re
import getpass
import urllib.parse

import pandas as pd
from PIL import Image
import gradio as gr
from google import genai
from datasets import load_dataset

MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")
MIN_API_INTERVAL = 5
_last_api_call_time = 0.0

def wait_for_api_slot():
    """Pace Gemini requests to respect the current free-tier request limit."""
    global _last_api_call_time

    elapsed = time.time() - _last_api_call_time

    if elapsed < MIN_API_INTERVAL:
        time.sleep(MIN_API_INTERVAL - elapsed)

    _last_api_call_time = time.time()

def load_api_key():
    """Load a Gemini API key without hard-coding it in the notebook."""
    key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY") or userdata.get("GOOGLE_API_KEY")
        except Exception:
            pass

    if not key:
        key = getpass.getpass("Gemini API key: ")

    if not key:
        raise ValueError("A Gemini API key is required to run the demo.")
    return key


CLIENT = genai.Client(api_key=load_api_key())

DISCLAIMER = """
<div style="
    background-color:#fff3cd;
    padding:12px;
    border-radius:6px;
    border:1px solid #d39e00;
    color:#5c4400;
    margin-bottom:12px;
">
  <b style="color:#8a2c0d;">⚠ ACADEMIC DEMO ONLY — NOT FOR CLINICAL USE.</b><br>
  <span style="color:#5c4400;">
    This proof-of-concept is not a medical device and must not be used
    for diagnosis, treatment or real clinical triage.
  </span>
</div>
"""

In [ ]:
# Optional API health check before launching the interface.
# Run this once. If it prints API_OK, the current key/model combination is responding.
try:
    test_response = CLIENT.models.generate_content(
        model=MODEL_NAME,
        contents="Reply with exactly: API_OK",
    )
    print("Model:", MODEL_NAME)
    print("Gemini test response:", (test_response.text or "").strip())
except Exception as exc:
    print("Model:", MODEL_NAME)
    print(type(exc).__name__)
    print(exc)


## 2. Model wrapper and dataset access

The simulation streams test samples from `mmenendezg/pneumonia_x_ray`. The model wrapper supports both text-only and image+text prompts and retains the original retry/backoff behaviour.

In [ ]:
def query_model(prompt, image=None):
    """
    Call Gemini with request pacing.

    If a 429 quota/rate-limit error occurs:
    - wait 65 seconds;
    - retry the same request once;
    - abort cleanly if the second attempt also receives 429.
    """

    contents = [prompt, image] if image is not None else prompt

    for quota_attempt in range(2):
        try:
            wait_for_api_slot()

            response = CLIENT.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
            )

            output = (response.text or "").strip()

            if not output:
                return "Error: Empty model response"

            return output

        except Exception as exc:
            message = str(exc)
            upper = message.upper()

            print(
                f"Gemini API attempt {quota_attempt + 1}/2 failed: "
                f"{type(exc).__name__}: {message}"
            )

            # Rate limit / quota
            if "429" in upper or "RESOURCE_EXHAUSTED" in upper:

                if quota_attempt == 0:
                    print(
                        "Rate limit reached. Waiting 65 seconds "
                        "before one retry..."
                    )
                    time.sleep(65)
                    continue

                return "Error: Gemini API quota/rate limit reached"

            # Temporary server-capacity failure
            if "503" in upper or "UNAVAILABLE" in upper:
                return "Error: Gemini API temporarily unavailable"

            # Timeout
            if (
                "504" in upper
                or "TIMEOUT" in upper
                or "DEADLINE" in upper
            ):
                return "Error: Gemini API request timed out"

            return f"Error: Gemini API unavailable ({type(exc).__name__})"

    return "Error: Gemini API quota/rate limit reached"


def get_patient_data(target_type="Random"):
    """Retrieve a chest X-ray sample from the public Hugging Face dataset."""
    try:
        ds = load_dataset(
            "mmenendezg/pneumonia_x_ray",
            split="test",
            streaming=True,
        )

        # Retained from the coursework implementation: randomly skip within the stream
        # and search for a requested class for the demonstration queue.
        for _ in range(100):
            skip = random.randint(0, 600)
            sample = next(iter(ds.skip(skip)))
            label = "Pneumonia" if sample["label"] == 1 else "Normal"

            if target_type == "Random" or target_type == label:
                return sample["image"], label, f"PID-{skip}"
    except Exception as exc:
        print(f"Dataset retrieval error: {type(exc).__name__}: {exc}")
        return None, None, None

    return None, None, None

# --------------------------------------------------------------------------
# Fixed portfolio evaluation set
# --------------------------------------------------------------------------
EVAL_CASES = None

def prepare_eval_cases(n_per_class=5, seed=42):
    """
    Build and cache a fixed balanced evaluation set.

    This avoids repeatedly reopening/skipping through the Hugging Face stream
    for every benchmark case. The same 10 X-rays are then reused by both the
    original workflow demonstration and the blinded benchmark, making the two
    runs easier to compare.
    """
    global EVAL_CASES

    if EVAL_CASES is not None:
        return EVAL_CASES

    print("Preparing balanced evaluation cases from Hugging Face...")

    try:
        ds = load_dataset(
            "mmenendezg/pneumonia_x_ray",
            split="test",
            streaming=True,
        )

        # Deterministic shuffle of the stream.
        ds = ds.shuffle(seed=seed, buffer_size=1000)

        buckets = {
            "Normal": [],
            "Pneumonia": [],
        }

        for idx, sample in enumerate(ds):
            label = "Pneumonia" if int(sample["label"]) == 1 else "Normal"

            if len(buckets[label]) < n_per_class:
                buckets[label].append({
                    "image": sample["image"],
                    "label": label,
                    "pid": f"PID-EVAL-{idx}",
                })

            if (
                len(buckets["Normal"]) >= n_per_class
                and len(buckets["Pneumonia"]) >= n_per_class
            ):
                break

        if (
            len(buckets["Normal"]) < n_per_class
            or len(buckets["Pneumonia"]) < n_per_class
        ):
            raise RuntimeError(
                "Could not collect enough balanced evaluation cases from the stream."
            )

        EVAL_CASES = buckets["Normal"] + buckets["Pneumonia"]

        # Deterministically mix the two classes so execution order is not
        # confounded with class label. Both benchmarks reuse this same order.
        rng = random.Random(seed)
        rng.shuffle(EVAL_CASES)

        print(
            "Evaluation set ready: "
            f"{len(buckets['Normal'])} Normal + "
            f"{len(buckets['Pneumonia'])} Pneumonia"
        )

        return EVAL_CASES

    except Exception as exc:
        print(f"Evaluation-set preparation failed: {type(exc).__name__}: {exc}")
        raise


## 3. Specialised agents

The five prompt-defined agents mirror the roles documented in the original project: History Generator, Context-Aware Vision, Medical Scribe, Triage Officer and Adviser.

## 3. Clinical-context simulation design

The autonomous workflow deliberately keeps the original `agent_history(truth)` design.

In the coursework simulation, the dataset label is used to seed a synthetic referral history. This is intended to emulate the fact that a radiologist normally receives a clinical indication or patient history before interpreting an image, allowing the project to demonstrate **contextual priming** across the multi-agent workflow.

This should be interpreted as a **workflow simulation**, not a blinded diagnostic experiment. The portfolio version therefore includes a separate **blinded benchmark**. In that path, the History Generator is bypassed, the Vision/Scribe/Triage agents receive only a neutral statement that no clinical history is available, and the dataset label is revealed only after the prediction has been produced for post-hoc scoring.


In [ ]:
def agent_adviser(history):
    prompt = f"""
Role: Symptom Guidance Agent in an academic healthcare AI demonstration.

The user has NOT supplied a chest X-ray. Therefore you must NOT diagnose
pneumonia or any other condition and must NOT claim that a disease is
present or absent.

SYMPTOMS PROVIDED:
"{history}"

Your task is to respond specifically to the symptoms provided rather than
giving the same generic response to every case.

First, briefly summarise the symptoms actually provided.

Then assign ONE academic demonstration guidance level:

DEMO GUIDANCE LEVEL: ROUTINE REVIEW
Use when the described symptoms appear mild and there are no clear
red-flag features in the text.

DEMO GUIDANCE LEVEL: PROMPT PROFESSIONAL REVIEW
Use when symptoms are persistent, worsening, or moderately concerning
but do not clearly describe an immediate emergency.

DEMO GUIDANCE LEVEL: URGENT PROFESSIONAL ASSESSMENT
Use when the supplied text describes concerning features such as severe
difficulty breathing, severe or worsening chest pain, confusion,
bluish lips/face, inability to speak normally because of breathlessness,
collapse, or similarly serious symptoms.

IMPORTANT:
- Base the response ONLY on the symptoms supplied.
- Do not infer the dataset label.
- Do not say "pneumonia", "normal X-ray", or claim a diagnosis.
- Do not invent symptoms that were not supplied.
- Different symptom descriptions should result in appropriately tailored
  guidance.
- This is an academic demonstration, not medical advice.
- If symptoms are ambiguous, acknowledge the uncertainty.

Use exactly this structure:

## Symptom-Only Guidance

**Symptoms provided:** [brief summary of the actual supplied symptoms]

**Demo guidance level:** [one of the three levels above]

**Why:** [one or two sentences referring specifically to the supplied symptoms]

**General supportive-care information:**
- [short relevant points]

**Seek urgent professional care if:**
- [short relevant red-flag points]

*Academic demonstration only — this is not a diagnosis or substitute for
professional medical assessment.*
"""

    return query_model(prompt)


def agent_history(truth):
    # Intentional workflow-simulation design:
    # the dataset label seeds a synthetic referral history to emulate the clinical
    # context a radiologist would normally receive before reviewing an image.
    # Because the context is label-seeded, this path must not be used as a blinded
    # diagnostic-accuracy benchmark.
    prompt = f"""Write a realistic two-sentence synthetic 'Clinical Indication' note for a chest X-ray referral. This is an academic simulation case labelled {truth}. Do not include identifying information."""
    return query_model(prompt)


def agent_vision(image, context):
    prompt = f"""
Role: Radiologist simulation for an academic demo.
Task: Describe visible findings in this chest X-ray.

INSTRUCTIONS:
- Use the clinical context to guide visual search, but only describe visible image evidence.
- Look for focal opacity, consolidation, infiltration and other obvious findings.
- If uncertain, state the uncertainty rather than inventing a finding.
- This output is not a clinical diagnosis.

CLINICAL CONTEXT: "{context}"
"""
    return query_model(prompt, image)


def agent_scribe(vision_text, history):
    prompt = f"""Format the following simulated notes into a concise structured chest X-ray report for an academic demonstration.

HISTORY: {history}
VISION NOTES: {vision_text}

Use headings for HISTORY, FINDINGS and IMPRESSION. Do not add findings that are absent from the vision notes."""
    return query_model(prompt)


def agent_triage(history, report):
    prompt = f"""
Role: Triage Officer simulation in an academic proof-of-concept.
Task: Apply the demonstration rules below to classify the generated text as URGENT or ROUTINE.

INPUT DATA:
1. Radiology findings/report: "{report}"
2. Patient symptoms/context: "{history}"

DEMONSTRATION TRIAGE RULES:
Rule 1:
If findings contain terms such as "Clear", "Normal", "Unremarkable" or "No acute findings", classify ROUTINE.
Exception: only override to URGENT if symptoms explicitly contain both "Shortness of breath" and "High fever".

Rule 2:
If findings contain terms such as "Pneumonia", "Opacity", "Consolidation", "Infiltration" or "Effusion", classify URGENT.

Rule 3:
If the report is inconclusive and symptoms are described as severe, classify URGENT for this demonstration.

OUTPUT FORMAT — FOLLOW EXACTLY:

DECISION: URGENT

or

DECISION: ROUTINE

After the DECISION line, provide:
DEMO GUIDANCE: A qualified clinician must review the case.
REASON: [one brief rule-based explanation]

IMPORTANT:
- The first line must contain exactly one final decision.
- Do not mention the alternative decision anywhere in the response.
- Do not write both URGENT and ROUTINE.
"""
    return query_model(prompt)

## 4. Orchestration helpers

In [ ]:
def build_history_string(c, m, b, t, p, ba, fa, ap, w, ex):
    symptoms = []
    if c:
        symptoms.append(f"Cough ({m})" if m else "Cough")
    if b:
        symptoms.append("Shortness of breath")
    if t:
        symptoms.append("High fever")
    if p:
        symptoms.append("Chest pain")
    if ba:
        symptoms.append("Body aches")
    if fa:
        symptoms.append("Fatigue")
    if ap:
        symptoms.append("Loss of appetite")
    if w:
        symptoms.append("Wheezing")

    base = "Patient reports: " + ", ".join(symptoms) + "." if symptoms else "General checkup."
    if ex:
        base += f" Notes: {ex}"
    return base


def generate_email_button(triage_text, pid, email_addr):
    """Create a local mailto draft. No email credentials are stored or transmitted by the notebook."""
    try:
        msg = triage_text.split("**INTERNAL REASONING SUMMARY:**")[-1].strip()
    except Exception:
        msg = "Review simulated case."

    decision = extract_triage_decision(triage_text)

    if decision == "URGENT":
        status = "URGENT DEMO ALERT"
    elif decision == "ROUTINE":
        status = "ROUTINE DEMO REPORT"
    else:
        status = "ANALYSIS UNAVAILABLE"

    recipient = email_addr if email_addr else "doctor@example.com"

    subject = urllib.parse.quote(f"{status}: Patient {pid}")
    body = urllib.parse.quote(
        f"Patient ID: {pid}\nStatus: {status}\n\nAcademic demo summary:\n{msg}\n\n"
        "This message was generated by an academic proof-of-concept and is not a clinical report."
    )

    return f"""
    <div style='background-color:#f4f4f9;padding:15px;border-radius:8px;border:1px solid #ddd;margin-top:10px;'>
      <b>Draft ready for:</b> {recipient}<br>
      <p style='font-size:0.9em;color:#555;'>{msg[:120]}...</p>
      <a href="mailto:{recipient}?subject={subject}&body={body}" target="_blank"
         style="background-color:#2b6cb0;color:white;padding:8px 15px;border-radius:5px;text-decoration:none;font-weight:bold;font-size:0.9em;">
         Open email draft
      </a>
    </div>
    """


def get_status_badge(text):
    decision = extract_triage_decision(text)

    if decision == "URGENT":
        return """
        <div style='background:#fee;color:#c00;padding:8px;border-radius:5px;
        text-align:center;font-weight:bold;'>
        STATUS: URGENT (DEMO)
        </div>
        """

    if decision == "ROUTINE":
        return """
        <div style='background:#efe;color:#080;padding:8px;border-radius:5px;
        text-align:center;font-weight:bold;'>
        STATUS: ROUTINE (DEMO)
        </div>
        """

    return """
    <div style='background:#fff3cd;color:#7a5200;padding:8px;border-radius:5px;
    text-align:center;font-weight:bold;'>
    STATUS: ANALYSIS UNAVAILABLE
    </div>
    """

import re

def extract_triage_decision(text):
    """Extract an explicit URGENT or ROUTINE triage decision robustly."""
    if not text:
        return "UNPARSEABLE"

    text = str(text).strip()

    patterns = [
        r"\bDECISION\s*:\s*(URGENT|ROUTINE)\b",
        r"\bTRIAGE(?:\s+STATUS|\s+DECISION)?\s*:\s*(URGENT|ROUTINE)\b",
        r"\bSTATUS\s*:\s*(URGENT|ROUTINE)\b",
        r"\bFINAL\s+(?:DECISION|ANSWER)\s*:\s*(URGENT|ROUTINE)\b",
        r"\b(?:DECISION|ANSWER)\s+(?:IS|=)\s*(URGENT|ROUTINE)\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return match.group(1).upper()

    # Last-resort check: use the final occurrence if only one class
    # is being asserted clearly.
    urgent_positions = [
        m.start() for m in re.finditer(r"\bURGENT\b", text, flags=re.IGNORECASE)
    ]
    routine_positions = [
        m.start() for m in re.finditer(r"\bROUTINE\b", text, flags=re.IGNORECASE)
    ]

    if urgent_positions and not routine_positions:
        return "URGENT"

    if routine_positions and not urgent_positions:
        return "ROUTINE"

    return "UNPARSEABLE"


BLINDED_CONTEXT = (
    "No clinical history is available for this blinded portfolio evaluation. "
    "Do not infer or assume the dataset label. Base the image findings only on visible X-ray evidence."
)


## 5. Gradio interface and workflows

The interface contains **Patient Intake**, **Simulation** and **Admin** tabs. The Admin tab now contains two deliberately separate evaluations: (1) the original label-seeded **workflow-consistency demonstration**, and (2) a portfolio-only **blinded benchmark** where the class label is withheld from all inference-stage agents.


In [ ]:
with gr.Blocks(title="Agentic Radiology Assistant") as app:
    with gr.Row():
        with gr.Column(scale=4):
            gr.Markdown("## Agentic Radiology Assistant")
        with gr.Column(scale=1):
            btn_shutdown = gr.Button("Shutdown System", variant="stop")

    gr.HTML(DISCLAIMER)

    with gr.Tabs():
        # Tab 1: Clinical Intake
        with gr.TabItem("Patient Intake"):
            mode = gr.Radio(["Has Image", "No Image"], label="Intake Type", value="Has Image")

            with gr.Column() as ui_img:
                with gr.Row():
                    img_input = gr.Image(type="pil", height=320)
                    with gr.Column():
                        gr.Markdown("### Clinical Data")
                        email_input = gr.Textbox(label="Demo Doctor Email", placeholder="doctor@example.com")

                        with gr.Row():
                            chk_cough = gr.Checkbox(label="Cough")
                            rad_mucus = gr.Radio(["Clear", "Color"], label="Mucus", interactive=False)
                        chk_cough.change(
                            lambda x: gr.update(interactive=x),
                            inputs=[chk_cough],
                            outputs=[rad_mucus],
                        )

                        with gr.Row():
                            chk_breath = gr.Checkbox(label="Shortness of breath")
                            chk_fever = gr.Checkbox(label="Fever")
                            chk_pain = gr.Checkbox(label="Chest pain")
                        with gr.Row():
                            chk_aches = gr.Checkbox(label="Body aches")
                            chk_tired = gr.Checkbox(label="Fatigue")
                            chk_appetite = gr.Checkbox(label="Loss of appetite")
                            chk_wheeze = gr.Checkbox(label="Wheezing")
                        txt_notes = gr.Textbox(label="Additional Notes")

                btn_analyze = gr.Button("Run Demo Pipeline", variant="primary")
                html_status = gr.HTML(label="Triage Decision")
                md_report = gr.Markdown(label="Generated Report")
                html_email = gr.HTML(visible=False)

            with gr.Column(visible=False) as ui_txt:
                txt_symptoms = gr.Textbox(label="Symptoms", lines=3)
                btn_advise = gr.Button("Get General Guidance")
                md_advice = gr.Markdown(label="Generated Guidance")

        # Tab 2: Autonomous Simulation
        with gr.TabItem("Simulation"):
            gr.Markdown("### Automated Workflow Simulation")
            with gr.Row():
                with gr.Column():
                    btn_sim = gr.Button("Run Simulation Case", variant="primary")
                    sim_email = gr.Textbox(label="Demo Doctor Email", placeholder="doctor@example.com")
                    sim_img = gr.Image(type="pil", height=300)
                    sim_truth = gr.Textbox(label="Dataset Label (shown for demonstration)")
                with gr.Column():
                    sim_status = gr.HTML(label="Demo Status")
                    sim_report = gr.Markdown(label="Generated Report")
                    sim_action = gr.HTML(visible=False)

        # Tab 3: Evaluation / validation demonstrations
        with gr.TabItem("Admin"):
            gr.Markdown(
                "### A. Original Workflow-Consistency Demonstration (Balanced n=10)\n"
                "**Design note:** the synthetic History Generator intentionally receives the dataset label "
                "to emulate referral context. This demonstrates the original contextual workflow and is "
                "**not** a blinded diagnostic-accuracy test."
            )
            btn_batch = gr.Button("Run Original Workflow Demo")
            md_results = gr.Markdown()
            df_logs = gr.DataFrame()

            gr.Markdown(
                "---\n"
                "### B. Blinded Portfolio Benchmark (Balanced n=10)\n"
                "This portfolio-only evaluation hides the dataset label from the History/Vision/Scribe/Triage "
                "inference path. The History Generator is bypassed, a neutral no-history statement is used, "
                "and the ground-truth label is revealed only **after** a triage decision is produced.\n\n"
                "For scoring only, the demonstration mapping is: **Pneumonia → URGENT** and **Normal → ROUTINE**. "
                "The result is reported as **blinded label-to-triage agreement**, not clinical diagnostic accuracy."
            )
            btn_blinded = gr.Button("Run Blinded Benchmark", variant="secondary")
            md_blinded_results = gr.Markdown()
            df_blinded_logs = gr.DataFrame()

    def toggle_ui(val):
        if val == "Has Image":
            return gr.update(visible=True), gr.update(visible=False)
        return gr.update(visible=False), gr.update(visible=True)

    mode.change(toggle_ui, inputs=[mode], outputs=[ui_img, ui_txt])

    def process_patient(img, mail, c, m, b, t, p, ba, fa, ap, w, ex):
        if img is None:
            return "Missing image", "", gr.update(visible=False)

        history = build_history_string(c, m, b, t, p, ba, fa, ap, w, ex)
        vision = agent_vision(img, history)
        if vision.startswith("Error:"):
          report = (
        f"**History:** {history}\n\n"
        f"**Findings:** Analysis could not be completed because the AI service "
        f"was temporarily unavailable.\n\n"
        f"*Technical status: {vision}*"
    )
          return (
        report,
        get_status_badge(vision),
        gr.update(visible=False),
    )
        decision = agent_triage(history, vision)

        report = f"**History:** {history}\n\n**Findings:** {vision}\n\n---\n\n{decision}"
        badge = get_status_badge(decision)

        email_button = gr.update(visible=False)
        if extract_triage_decision(decision) == "URGENT":
            email_button = gr.update(
                value=generate_email_button(decision, "USER", mail),
                visible=True,
            )

        return report, badge, email_button

    btn_analyze.click(
        process_patient,
        inputs=[
            img_input, email_input, chk_cough, rad_mucus, chk_breath,
            chk_fever, chk_pain, chk_aches, chk_tired, chk_appetite,
            chk_wheeze, txt_notes,
        ],
        outputs=[md_report, html_status, html_email],
    )

    btn_advise.click(agent_adviser, inputs=[txt_symptoms], outputs=[md_advice])

    def process_sim(email, progress=gr.Progress()):
        progress(0.1, desc="Acquiring data...")
        while True:
            image, truth, pid = get_patient_data("Random")
            if image is not None:
                break

        progress(0.3, desc="Generating synthetic history...")
        history = agent_history(truth)

        progress(0.5, desc="Analysing scan...")
        vision = agent_vision(image, history)
        report_raw = agent_scribe(vision, history)

        progress(0.8, desc="Applying demo triage rules...")
        decision = agent_triage(history, report_raw)

        full_report = f"### CASE: {pid}\n\n{report_raw}\n\n---\n\n{decision}"
        badge = get_status_badge(decision)

        email_button = gr.update(visible=False)
        if extract_triage_decision(decision) == "URGENT":
            email_button = gr.update(
                value=generate_email_button(decision, pid, email),
                visible=True,
            )

        return image, truth, full_report, badge, email_button

    btn_sim.click(
        process_sim,
        inputs=[sim_email],
        outputs=[sim_img, sim_truth, sim_report, sim_status, sim_action],
    )

    def abort_benchmark_result(title, reason, logs, columns=None):
        """Return a clear partial-result table when the external AI service fails."""
        if logs:
            df = pd.DataFrame(logs)
        else:
            df = pd.DataFrame(columns=columns or [])

        summary = (
            f"## {title} aborted\n\n"
            f"**Reason:** {reason}\n\n"
            f"Completed cases before abort: **{len(df)}**\n\n"
            "No benchmark performance percentage is reported because the full "
            "10-case evaluation did not complete successfully. Please retry when "
            "the Gemini API is available and within quota."
        )
        return summary, df


    def run_batch(progress=gr.Progress()):
        """Demonstrate workflow consistency on the fixed 10-case queue.

        DESIGN NOTE: agent_history intentionally receives the true class to seed
        synthetic referral context, representing the clinical history available
        upstream in the simulated workflow. This is therefore a workflow
        demonstration, not a blinded diagnostic-accuracy benchmark.
        """
        logs = []
        cases = list(prepare_eval_cases())

        for i, case in enumerate(cases):
            truth = case["label"]
            image = case["image"]
            pid = case["pid"]
            base = i / len(cases)

            progress(
                base + 0.01,
                desc=f"Case {i + 1}/10 — generating history..."
            )
            synthetic_history = agent_history(truth)

            if synthetic_history.startswith("Error:"):
                return abort_benchmark_result(
                    "Original workflow-consistency benchmark",
                    f"History Generator failed: {synthetic_history}",
                    logs,
                    columns=[
                        "ID",
                        "Dataset label",
                        "Workflow decision",
                        "Expected decision",
                        "Consistency",
                    ],
                )

            progress(
                base + 0.04,
                desc=f"Case {i + 1}/10 — analysing X-ray..."
            )
            vision_output = agent_vision(image, synthetic_history)

            if vision_output.startswith("Error:"):
                return abort_benchmark_result(
                    "Original workflow-consistency benchmark",
                    f"Vision Agent failed: {vision_output}",
                    logs,
                    columns=[
                        "ID",
                        "Dataset label",
                        "Workflow decision",
                        "Expected decision",
                        "Consistency",
                    ],
                )

            progress(
                base + 0.07,
                desc=f"Case {i + 1}/10 — applying triage..."
            )
            triage_output = agent_triage(synthetic_history, vision_output)

            if triage_output.startswith("Error:"):
                return abort_benchmark_result(
                    "Original workflow-consistency benchmark",
                    f"Triage Agent failed: {triage_output}",
                    logs,
                    columns=[
                        "ID",
                        "Dataset label",
                        "Workflow decision",
                        "Expected decision",
                        "Consistency",
                    ],
                )

            decision = extract_triage_decision(triage_output)
            expected = "URGENT" if truth == "Pneumonia" else "ROUTINE"

            if decision == "UNPARSEABLE":
                consistency = "UNSCORABLE"
            elif decision == expected:
                consistency = "PASS"
            else:
                consistency = "MISMATCH"

            logs.append({
                "ID": pid,
                "Dataset label": truth,
                "Workflow decision": decision,
                "Expected decision": expected,
                "Consistency": consistency,
            })

            progress(
                (i + 1) / len(cases),
                desc=f"Case {i + 1}/10 complete"
            )

        df = pd.DataFrame(logs)

        passes = int((df["Consistency"] == "PASS").sum())
        mismatches = int((df["Consistency"] == "MISMATCH").sum())
        unscorable = int((df["Consistency"] == "UNSCORABLE").sum())
        parseable = passes + mismatches

        parse_rate = round((parseable / len(df)) * 100, 1)
        strict_rate = round((passes / len(df)) * 100, 1)
        agreement = round((passes / parseable) * 100, 1) if parseable else 0.0

        summary = (
            f"## Original workflow-consistency results\n\n"
            f"- Strict end-to-end passes: **{passes}/{len(df)} ({strict_rate}%)**\n"
            f"- Decision parse coverage: **{parseable}/{len(df)} ({parse_rate}%)**\n"
            f"- Agreement among parseable decisions: "
            f"**{passes}/{parseable} ({agreement}%)**\n"
            f"- Genuine mismatches: **{mismatches}**\n"
            f"- Unscorable outputs: **{unscorable}**\n\n"
            "*This is a workflow-consistency demonstration, not a blinded "
            "diagnostic-accuracy measurement.*"
        )

        return summary, df


    def run_blinded_benchmark(progress=gr.Progress()):
        """Run the fixed 10-case blinded label-to-triage agreement benchmark.

        The dataset label is hidden from every inference-stage agent and used
        only after inference for post-hoc scoring.
        """
        logs = []
        cases = list(prepare_eval_cases())

        for i, case in enumerate(cases):
            image = case["image"]
            real_label = case["label"]
            pid = case["pid"]
            base = i / len(cases)

            progress(
                base + 0.02,
                desc=f"Blinded case {i + 1}/10 — analysing X-ray..."
            )
            vision_output = agent_vision(image, BLINDED_CONTEXT)

            if vision_output.startswith("Error:"):
                return abort_benchmark_result(
                    "Blinded portfolio benchmark",
                    f"Vision Agent failed: {vision_output}",
                    logs,
                    columns=[
                        "ID",
                        "Dataset label (revealed after inference)",
                        "Blinded triage decision",
                        "Expected demo mapping",
                        "Agreement",
                    ],
                )

            progress(
                base + 0.05,
                desc=f"Blinded case {i + 1}/10 — generating report..."
            )
            report_output = agent_scribe(
                vision_output,
                BLINDED_CONTEXT
            )

            if report_output.startswith("Error:"):
                return abort_benchmark_result(
                    "Blinded portfolio benchmark",
                    f"Medical Scribe failed: {report_output}",
                    logs,
                    columns=[
                        "ID",
                        "Dataset label (revealed after inference)",
                        "Blinded triage decision",
                        "Expected demo mapping",
                        "Agreement",
                    ],
                )

            progress(
                base + 0.08,
                desc=f"Blinded case {i + 1}/10 — applying triage..."
            )
            triage_output = agent_triage(
                BLINDED_CONTEXT,
                report_output
            )

            if triage_output.startswith("Error:"):
                return abort_benchmark_result(
                    "Blinded portfolio benchmark",
                    f"Triage Agent failed: {triage_output}",
                    logs,
                    columns=[
                        "ID",
                        "Dataset label (revealed after inference)",
                        "Blinded triage decision",
                        "Expected demo mapping",
                        "Agreement",
                    ],
                )

            decision = extract_triage_decision(triage_output)

            # Hidden label is used only here, after inference, for scoring.
            expected_demo_decision = (
                "URGENT" if real_label == "Pneumonia" else "ROUTINE"
            )

            if decision == "UNPARSEABLE":
                agreement_status = "UNSCORABLE"
            elif decision == expected_demo_decision:
                agreement_status = "PASS"
            else:
                agreement_status = "MISMATCH"

            logs.append({
                "ID": pid,
                "Dataset label (revealed after inference)": real_label,
                "Blinded triage decision": decision,
                "Expected demo mapping": expected_demo_decision,
                "Agreement": agreement_status,
            })

            progress(
                (i + 1) / len(cases),
                desc=f"Blinded case {i + 1}/10 complete"
            )

        df = pd.DataFrame(logs)

        passes = int((df["Agreement"] == "PASS").sum())
        mismatches = int((df["Agreement"] == "MISMATCH").sum())
        unscorable = int((df["Agreement"] == "UNSCORABLE").sum())
        parseable = passes + mismatches

        parse_rate = round((parseable / len(df)) * 100, 1)
        agreement = round((passes / parseable) * 100, 1) if parseable else 0.0

        pneumonia_rows = df[
            df["Dataset label (revealed after inference)"] == "Pneumonia"
        ]
        normal_rows = df[
            df["Dataset label (revealed after inference)"] == "Normal"
        ]

        pneumonia_passes = int((pneumonia_rows["Agreement"] == "PASS").sum())
        normal_passes = int((normal_rows["Agreement"] == "PASS").sum())

        summary = (
            f"## Blinded label-to-triage agreement results\n\n"
            f"- Decision parse coverage: **{parseable}/{len(df)} ({parse_rate}%)**\n"
            f"- Agreement among parseable decisions: "
            f"**{passes}/{parseable} ({agreement}%)**\n"
            f"- Pneumonia → URGENT passes: **{pneumonia_passes}/5**\n"
            f"- Normal → ROUTINE passes: **{normal_passes}/5**\n"
            f"- Genuine mismatches: **{mismatches}**\n"
            f"- Unscorable outputs: **{unscorable}**\n\n"
            "*The dataset label is hidden from all inference-stage agents and "
            "used only after prediction for scoring. This is a small academic "
            "proof-of-concept benchmark and is not clinical diagnostic accuracy.*"
        )

        return summary, df


    btn_batch.click(run_batch, outputs=[md_results, df_logs])
    btn_blinded.click(
        run_blinded_benchmark,
        outputs=[md_blinded_results, df_blinded_logs],
    )
    btn_shutdown.click(app.close)

## 6. Blinded portfolio benchmark

The **Blinded Portfolio Benchmark** is an addition made for this GitHub portfolio version; it was not part of the original submitted coursework.

For each balanced test case:

1. the loader uses the dataset label only to obtain five `Normal` and five `Pneumonia` samples;
2. the label is **not passed to any inference-stage agent**;
3. the History Generator is bypassed;
4. the Vision Agent receives the X-ray plus a neutral statement that no clinical history is available;
5. the Scribe formats the image findings;
6. the Triage Agent returns `URGENT` or `ROUTINE`;
7. only then is the hidden dataset label revealed for scoring.

The scoring mapping is deliberately limited to this demonstration: `Pneumonia → URGENT` and `Normal → ROUTINE`. The output is therefore called **blinded label-to-triage agreement**, not medical diagnostic accuracy.


## 7. Launch

In Colab, Gradio may automatically create a temporary share link. Do not commit temporary share URLs to the repository.

In [ ]:
# Prepare the fixed balanced 10-case evaluation set once before launching the UI.
# Both Admin benchmarks reuse these same X-rays.
EVAL_CASES = prepare_eval_cases(n_per_class=5, seed=42)


In [ ]:
print("System ready. Launching academic demo interface...")

app.launch(
    debug=True,
    height=1000
)